# Gust Loads

In this section a gust loads analyses is going to be conducted. The  simulation set-up in <em>Loads Kernel</em> was already shown in detail in a previous notebook. Hence, the focus will be more in the results and post processing. In order to showcase the gust load analyses capabilities of <em>Loads Kernel</em> only one gust gradeint was simulated in the first example. This is justified with the large amount of time necessary to perform the simulation with all the different gust gradients. Nevertheless, the simulation with all the different gust gradients is shown and it is explained under the name 'Advanced Simulation' below.

## Simulation Set-up

The simulation set-up is similar to the one done for the trim and maneuver load analyses with the only exception being the last part of the set-up regarding the self.trimcase and the self.simcase. To analyze gust loads, horizontal level flight (n<sub>Z</sub>=1) FL000 at trim conditions was selected. Again the mass configuration M3 was chosen. The gust gradient selected for this example is the gust gradient of 23 meters, which corresponds to the limit case. In the simulation a time step of 0.01 seconds and a simulation time of 2 seconds were utilized. The gust parameters were obtained in accordance with CS-25 {cite}`cs25`.

In [ ]:
class jcl:

    def __init__(self):
        
        self.trimcase = [{'desc': 'CC.M3.OVCFL000.level', # Descriptive string of the maneuver case
                          # Kind of trim condition, blank for trim about all three axes, for more trim conditions see
                          # trim_conditions.py
                          'maneuver': '',
                          # Subcase ID number, for Nastran in acending order
                          'subcase': 1,
                          # Setting of the operational point
                          # The flight speed is given by the Mach number
                          'Ma': tas2Ma(70.0, 0.0),
                          # Aero key
                          'aero': 'VC',
                          # Atmo key
                          'altitude': 'FL000',
                          # Mass key
                          'mass': 'M3',
                          # Load factor Nz
                          'Nz': 1.0,
                          # Velocities and accelerations given in ISO 9300 coordinate system (right-handed, forward-right-down)
                          # Roll rate in rad/s
                          'p': 0.0 / 180.0 * np.pi,
                          # Pitch rate in rad/s
                          'q': 0.0 / 180.0 * np.pi,
                          # Yaw rate in rad/s
                          'r': 0.0,
                          # Roll acceleration in rad/s^2
                          'pdot': 0.0 ,
                          # Pitch acceleration in rad/s^2
                          'qdot': 0.0,
                          # Yaw acceleration in rad/s^2
                          'rdot': 0.0,
                          }]
        
        self.simcase = [{'dt': 0.01,  # Time step size of the output in [s]
                         # Final simulation time  in [s]
                         't_final': 2.0,
                         # True or False, enables 1-cosine gust according to CS-25
                         'gust': True,
                         # Gust gradient H (half gust length) in [m]
                         'gust_gradient': 23.0,
                         # Orientation of the gust in [deg], 0/360 = gust from bottom, 180 = gust from top,
                         # 90 = gust from the right, 270 = gust from the left, arbitrary values possible
                         # (rotation of gust direction vector about Nastran's x-axis pointing backwards)
                         'gust_orientation': 0,
                         # Gust parameters according to CS-25 to calculate the gust velocity
                         # MD is the dive speed in Mach
                         'gust_para': {'Z_mo': 8046.72, 'MLW': 11793.40, 'MTOW': 11883.98, 'MZFW': 10594.47, 'MD': 0.334,
                                       'T1': 0.0},
                         # Alternatively, give gust velocity / Vtas directly
                         # 'WG_TAS': 0.1,
                         # True or False, enables continuous turbulence excitation
                         'turbulence': False,
                         # True or False, calculates limit turbulence according to CS-25
                         'limit_turbulence': False,
                         # True or False, enables playback of control surface signals via efcs
                         'cs_signal': False,
                         # True or False, enables a generic controller e.g. to maintain p, q and r
                         'controller': False,
                         # True or False, enables a generic landing gear
                         'landinggear': False,
                         # True or False, enables calculation of rigid and elastic derivatives
                         'derivatives': False,
                         # True or False, enables flutter check with k, ke or pk method
                         'flutter': False,
                         }]

In order to obtain the results the following 'launch.py' file is run. The results can be found in the folder 'DC-3_results'.

In [ ]:
from loadskernel import program_flow

# Here you launch the Loads Kernel with your job
k = program_flow.Kernel('jcl_dc3_gust_H23', pre=True, main=True, post=True, test=False,
                        path_input='./DC3_model/JCLs',
                        path_output='./DC3_results')
k.run()

## Results

As it was done for the analyses of the maneuver loads, two-dimensional load envelopes were constructed. Again, the M<sub>x</sub>/M<sub>y</sub> load envelope is going to be interpreted. In the following figure the torsional and bending moments envelope is shown for the left wing root. First, one can see that the envelope shape resembles a certain direct correlation between the bending and torsional moments. Second, one can see that the highest negative bending moments  M<sub>x</sub> and highest negative torsional moments M<sub>y</sub> are reached by the gust load at the time instants between 0.36 and 0.48 seconds. The highest positive bending moments  M<sub>x</sub> and the highest positive torsional moments M<sub>y</sub> are reached by the gust load at the time instants close to 1 second.

<img src="./images/MyMx_gust.png" width="400" alt="DC3">

*Bending moment M<sub>x</sub> and torsional moment M<sub>y</sub> at the wing root.*

Once again, one-dimensional envelopes along the wing span were studied to provide insights into the evolution of section loads across the wing. The next figure illustrates the envelope of the bending moment M<sub>x</sub> along the wingspan. The highest positive bending moment arise from the gust load at the time instants between 0.50 and 0.54 seconds. On the other hand, the highest negative bending moment arises from the gust load at the time instants between 0.87 and 0.88 seconds.

<img src="./images/Mx_y_gust.png" width="400" alt="DC3">

*Bending moment M<sub>x</sub> along the wing span.*

Furthermore, in order to have a deeper understanding of the gust loads, the forces and moments of the different gust profiles were plotted over time. Observing, the bending moments M<sub>x</sub> over time for the gust gradients H=23m at the wing root (figure below), it can be seen that the magnitude of the positive peak is very close to the negative peak, which was unexpected. The explanation for this comes further on in the Advanced Simulation section.

<img src="./images/Mx_time.png" width="400" alt="DC3">

*Bending moment M<sub>x</sub> over time at the wing root.*

## Advanced Simulation

In this section a more complete gust load analyses is presented. In the DC-3 case, for the gust load analyses, a gust gradient of 9, 16, 23, 30, 37, 51, 65, 79, 93 and 107 meters were selected in accordance with CS-25.341 {cite}`cs25`. In every simulation a time step of 0.01 seconds and a simulation time of 3 seconds were utilized. The gust parameters were obtained in accordance with CS-25 too.

To run the simulation the 'launch.py' file is run. The results can be fouund in the folder 'DC-3_results'.

In [ ]:
from loadskernel import program_flow

# Here you launch the Loads Kernel with your job
k = program_flow.Kernel('jcl_dc3_gust', pre=True, main=True, post=True, test=False,
                        path_input='./DC3_model/JCLs',
                        path_output='./DC3_results')
k.run()

## Results Advanced Simulation

To study the gust loads the (1-cos) Discrete Gust Model was implemented in <em>Loads Kernel</em>. The gust analyses was performed in the same conditions as the maneuvers at True Airspeed (TAS) of 70 m/s and Flight Level FL000 at trim conditions. In accordance to CS-25.341 {cite}`cs25` the gust profiles at sea level were obtained, and are exposed in the next figure.

<img src="./images/gust_shapes.png" width="350" alt="DC3">

*Gust shapes at sea-level according to CS-25.*

To have a more detailed understanding of the gust shapes, the gust gradients and their corresponding peak velocities are listed in the following table. Concerning the orientation of the gust, only vertical gusts are going to be studied.

<img src="./images/gust_table.png" width="500" alt="DC3">

*Gust gradients and corresponding peak velocities.*

As it was done for the analyses of the maneuver loads, two-dimensional load envelopes were constructed. Again, the M<sub>x</sub>/M<sub>y</sub> load envelope is going to be interpreted. In the figure below the torsional and bending moments envelope is shown for the left wing root. First, one can see that the envelope shape resembles a certain direct correlation between the bending and torsional moments. Second, one can see that the highest negative bending moments M<sub>x</sub> are reached by the gust loads with gust gradients H=16, 23, 30 and 37 m, while the highest negative torsional moments M<sub>y</sub> are reached by the gust loads with gust gradient H=9 m. The highest positive bending moments M<sub>x</sub> are reached by the gust load with gust gradient H=51 m, this gust load corresponds to the smallest negative torsional moments M<sub>y</sub>. Third, both maneuver and gust envelopes are very close to each other, which is an indicator for well-selected load cases which harmonize with each other in the sense that there are no extreme loads cases which dominate the design. That can be concluded comparing the figure below with the analogous figure obtained in the manuever load analyses.

<img src="./images/MxMy_gust_adv.png" width="400" alt="DC3">

*Bending moment M<sub>x</sub> and torsional moment M<sub>y</sub> at the wing root.*

Once again, one-dimensional envelopes along the wing span were studied to provide insights into the evolution of section loads across the wing. The following figure illustrates the envelope of the bending moment M<sub>x</sub> along the wing span. Comparing the results between the maneuver and gust loads, one can see that magnitude of the values is the same. Due to the fact that the results are symmetric, the conclusions drawn from the right wing (y>0) are valid for the left wing. In the inboard part of the right wing, the highest positive bending moment arise from the gust load with gust gradient H=23 m. For the outboard section the highest values correspond to the gust load with gust gradient H=16 m, except for the last three monitoring stations closer to the tip. For the monitoring station located at the tip the
highest value is for the gust load with gust gradient H=51 m and for the other two is for the gust load with gust gradient H=9 m. On the other hand, the highest negative bending moment arises from the gust load with gust gradient H=51 m for the entire wing span, except for the tip for which the highest value is for the gust load with gust gradient H=37 m.

<img src="./images/Mx_y_gust_adv.png" width="400" alt="DC3">

*Bending moment M<sub>x</sub> along the wing span.*

Furthermore, in order to have a deeper understanding of the gust loads, the forces and moments of the different gust profiles were plotted over time. The time step chosen was 0.01 seconds which is a step size with a good balance between accuracy and computational efficiency. It maintains acceptable results and at the same time is large enough to not be very computationally demanding. The interval time selected
for the gust analyses was 3 seconds, which is more than enough to capture the several peaks for each gust profile. Observing, the bending moments M<sub>x</sub> over time for the different gust gradients at the wing root (figure below), it can be seen that the magnitude of the positive peaks is very close to the negative peaks, which was unexpected. Moreover, the highest positive peak is reached for a gust gradient of H=23 m. Empirical data collected in the 1940s shows that the highest gust excitations on legacy aircraft occur at a gradient corresponding to 12.5 spatial chord lengths on average. Being the mean aerodynamic chord c=3.508 m, the highest positive peak should have been reached around a gust gradient of H=44 m, almost the value obtained.

<img src="./images/Mx_t_gust_adv.png" width="450" alt="DC3">

*Bending moment M<sub>x</sub> along the wing span.*

It was tought that the source was possible a coupling between the gusts and one of the longitudinal flight mechanics modes, namely the short period. One way to eliminate this coupling is to decreased the distance between the aerodynamic center and the center of gravity, so the static margin. The aircraft model has a very high static margin in the M3 mass configuration, which was reduced to approximately 10%. This resulted in the magnitude of the positive and negative peaks of the moments/forces over time plots to change so that the magnitude of these peaks was then much different. On top of that, the highest peak was then arising from a gust with a gust gradient between H=37 m
and H=51 m. This means the empirical predicted gust gradient value of H=44 m falls in this range. This goes to show that the unexpected gust load results obtained are justified by this gust and short period coupling that happens for gust profiles with small gust gradients.

For additional information about the meanuver load results please refer to {cite}`Carvalho2024`.